# DCEA Hands-On Exercise in R: Bowel Cancer Screening Reminders, on Real Public Data

**Distributional Cost-Effectiveness Analysis (DCEA)**: 40-minute hands-on companion to the workshop deck.

> **Colab:** this notebook should open in Colab's **R** runtime automatically. If the first cell complains about R syntax, choose `Runtime → Change runtime type → R`. Everything uses base R, so there is nothing to install.

**The question.** Jack lives in the most deprived fifth of England and can expect about 63 years in full health; Jill, in the least deprived fifth, about 75. England's Bowel Cancer Screening Programme (BCSP) has lower uptake in deprived areas like Jack's. Should the NHS send a cheap **universal reminder** letter to everyone invited, or a more intensive **targeted reminder** only to people in the most deprived areas? Universal reaches more people. Targeted reaches the people with the least health. DCEA makes that trade-off explicit.

**The data. Everything here is public:**

| Source | What we use | Licence |
|---|---|---|
| York Health Equity Impact Calculator (Love-Koh, Schneider & Cookson 2022), [github.com/bitowaqr/dcea](https://github.com/bitowaqr/dcea) | Baseline quality-adjusted life expectancy (QALE) by IMD quintile; hospital episodes by ICD-10 code × age × IMD; opportunity-cost distributions | MIT, acknowledge authors |
| Asaria, Griffin & Cookson (2016), *Medical Decision Making* 36:8–19 | Screening uptake by deprivation (Table 2), reminder costs and effects (Table 3), and the 20-subgroup QALE data for the fairness exercise (Tables 5–6) | Published tables |
| Chetty et al. (2016), *JAMA*, [healthinequality.org](https://healthinequality.org/data/) | US life expectancy at 40 by sex × income percentile (stretch task) | CC0 |

**How this notebook works**
- Runs top-to-bottom on stock Google Colab: no installs, no uploads. Data is loaded from this workshop's GitHub repository, falling back to the original sources (pinned to a fixed version).
- `Runtime → Run all` completes without errors before you write anything: the TODO self-checks fail gracefully.
- Each section: a short **TODO** cell (the hands-on part), a **self-check**, then a **Solution** you can run if you're stuck or short on time. Later sections use the solution functions, so you can skip ahead at any point.
- Time budget: ~5 / 12 / 7 / 7 / 9 minutes for Sections 1–5. Aim to get a number or a plot out of every section, not to be perfect.

**New to cost-effectiveness analysis? Five terms you'll need**

| Term | Meaning |
|---|---|
| **QALY** | Quality-adjusted life year: one year in full health. Two years at half health is also 1 QALY. |
| **QALE** | Quality-adjusted life expectancy: the QALYs a person can expect over their whole life. |
| **ICER** | Incremental cost-effectiveness ratio: extra cost ÷ extra QALYs, compared with the alternative. |
| **Threshold** | The most the health system will pay per QALY (NICE in England: £20,000–30,000). |
| **Health opportunity cost / net health benefit** | Money spent here isn't spent on other care, so someone else loses health: cost ÷ threshold QALYs. Net health benefit = QALYs gained − that loss. Positive means population health goes up. |

> ⚠️ **Your numbers will not match the paper.** This notebook applies the paper's *method* to *current public data*. Only Section 3 (fairness adjustment) uses the paper's own subgroup data and reproduces its published coefficients.

In [ ]:
options(digits = 4, width = 110, repr.plot.width = 8, repr.plot.height = 4.5)

IMD_COLOURS   <- c("#7a0177", "#c51b8a", "#f768a1", "#fa9fb5", "#fcc5c0")  # IMD1 (most deprived) ... IMD5
STRAT_COLOURS <- c(universal = "#1b7837", targeted = "#762a83")

# Data sources, pinned so results are reproducible
YORK_COMMIT <- "ddb3ea29c11ba2041cea2f2dc6856899c37b5282"   # bitowaqr/dcea, 22 Oct 2023
YORK_URL    <- paste0("https://raw.githubusercontent.com/bitowaqr/dcea/", YORK_COMMIT, "/data/")
CHETTY_URL  <- "https://healthinequality.org/dl/health_ineq_online_table_1.csv"

# This workshop's GitHub copy of the data (a backup if an original source is unavailable)
GITHUB_REPO   <- "YOUR-USERNAME/dcea-workshop"               # <- set to "user/repo" after uploading
REPO_DATA_URL <- paste0("https://raw.githubusercontent.com/", GITHUB_REPO, "/main/data/")

read_lines_from <- function(src) {
  con <- if (grepl("^https?://", src))
    # some servers (e.g. healthinequality.org) refuse requests without a browser user-agent
    url(src, headers = c("User-Agent" = "Mozilla/5.0"), encoding = "UTF-8") else file(src, encoding = "UTF-8")
  on.exit(close(con))
  readLines(con, warn = FALSE)
}

load_csv <- function(url, local_name) {
  # Try, in order: a local ./data/ copy, this workshop's GitHub copy, then the original source.
  for (src in c(file.path("data", local_name), paste0(REPO_DATA_URL, local_name), url)) {
    if (!grepl("^https?://", src) && !file.exists(src)) next
    lines <- tryCatch(suppressWarnings(read_lines_from(src)), error = function(e) NULL)
    if (length(lines)) {
      lines[1] <- sub("^﻿", "", lines[1])                # drop the byte-order mark some files carry
      return(read.csv(text = lines, stringsAsFactors = FALSE))
    }
  }
  stop("Could not load ", local_name, " from any source")
}

qale_imd <- load_csv(paste0(YORK_URL, "qale_pop_imd.csv"), "york_qale_pop_imd.csv")
hes      <- load_csv(paste0(YORK_URL, "icd_imd.csv"), "york_icd_imd.csv")
hoc_dist <- load_csv(paste0(YORK_URL, "hoc_distribution.csv"), "york_hoc_distribution.csv")
cat("loaded:", dim(qale_imd), "|", dim(hes), "|", dim(hoc_dist), "\n")

## Section 1: Baseline health distribution (~5 min)

DCEA starts from the **baseline distribution of health**: how much lifetime health (QALE, quality-adjusted life expectancy at birth) each social group can expect *before* any intervention.

The York calculator provides this for England by **IMD quintile** (Index of Multiple Deprivation, a small-area measure). Each quintile holds ~10.7 million people.

> 🔁 **Watch the coding direction.** In the York data **IMD1 = most deprived**. In the Asaria et al. paper, Q1 = *least* deprived. Mixing these up silently reverses every equity conclusion. It is one of the most common real-world DCEA bugs.

Run the cell and read the plot. No TODO here.

In [ ]:
imd_labels <- c("IMD1\n(most deprived)", "IMD2", "IMD3", "IMD4", "IMD5\n(least deprived)")
mean_qale <- weighted.mean(qale_imd$qale, qale_imd$pop)
gap <- qale_imd$qale[5] - qale_imd$qale[1]

print(qale_imd[, c("imd", "qale", "pop")])
cat(sprintf("\nPopulation mean QALE: %.2f   |   gap IMD5 - IMD1: %.2f QALYs\n", mean_qale, gap))

barplot(qale_imd$qale, names.arg = imd_labels, col = IMD_COLOURS, ylim = c(55, 80), xpd = FALSE,
        ylab = "QALE at birth (QALYs)", main = "Baseline health distribution, England (York calculator data)")
abline(h = mean_qale, lty = 2)
legend("topleft", lty = 2, legend = sprintf("population mean %.1f", mean_qale), bty = "n")

> A child born in the most deprived fifth of England can expect about **12 fewer healthy years** than one in the least deprived fifth. This is the distribution our screening reminders will (very slightly) shift.

## Section 2: Who benefits, and who pays? (~12 min)

To model the distribution of an intervention's effects we need, for each IMD group:

1. **Need**: where does the disease fall? We use hospital episodes for bowel cancer (ICD-10 **C18–C20**) among people aged 60–74 (the screening age range) from the York hospital-episode file.
2. **Uptake**: who actually completes the screening test? From the paper's Table 2 (averaged over sex and ethnicity, and re-coded to York's IMD direction).
3. **Effect and cost** per extra person screened, derived from the paper.
4. **Opportunity cost**: the money spent displaces other NHS care. Whose health does that come from? The York data offers three scenarios: `flat` (equal shares, as in the paper), `moderate`, and `steep` (more falls on the deprived).

### 2a. Data check first: hospital episodes

Run the next cell. **Look closely at the IMD3 and IMD4 columns.** What do you notice?

In [ ]:
bowel <- subset(hes, icd %in% c("C18", "C19", "C20") & age %in% c(60, 70))
episodes_wide <- reshape(bowel, idvar = c("icd", "age"), timevar = "imd", direction = "wide")
print(episodes_wide, row.names = FALSE)

# Is IMD4 just a copy of IMD3 across the *whole* file?
all_wide <- reshape(hes, idvar = c("icd", "age"), timevar = "imd", direction = "wide")
cat(sprintf("\nShare of ALL ICD x age rows where IMD3 == IMD4 exactly: %.0f%%\n",
            100 * mean(all_wide$episodes.3 == all_wide$episodes.4)))
cat(sprintf("Share of rows where IMD1 == IMD2 exactly (for comparison): %.0f%%\n",
            100 * mean(all_wide$episodes.1 == all_wide$episodes.2)))
cat(sprintf("Share of cells equal to 5 (small-number floor): %.0f%%\n", 100 * mean(hes$episodes == 5)))

> **What we found:** in the published York file, the IMD4 column is identical to IMD3 in **every** row, which looks like a copy error in the source file. About a third of cells are also floored at 5, a typical small-number suppression. Public data is rarely perfect. Here the practical impact is small (IMD3 and IMD4 have similar bowel cancer burdens), so we carry on but **record it as a limitation**. Always check before you model.

### 2b. Build the input table (pre-built)

In [ ]:
# Need: each IMD group's share of bowel cancer episodes, ages 60-74
need <- tapply(bowel$episodes, bowel$imd, sum)
need_share <- as.numeric(need / sum(need))

# Uptake by IMD from Asaria et al. (2016) Table 2, population-weighted over sex x ethnicity.
# The paper's Q1 = least deprived, so paper Q1..Q5 -> York IMD5..IMD1.
fobt_uptake <- c(0.450, 0.552, 0.611, 0.654, 0.691)   # IMD1..IMD5: completes the gFOBT kit
colo_uptake <- c(0.763, 0.781, 0.796, 0.836, 0.856)   # IMD1..IMD5: attends follow-up colonoscopy

COHORT <- 1e6                        # people invited per screening round (the paper's convention)
inputs <- data.frame(imd = 1:5, invitees = COHORT / 5,   # IMD quintiles are equal-sized
                     need_share = need_share, fobt_uptake = fobt_uptake, colo_uptake = colo_uptake)

# ---- ASSUMPTIONS (change these and re-run to explore) -----------------------
PARAMS <- list(
  # From the paper: standard screening gains 0.0436 QALYs per invitee (before opportunity
  # cost) at 59% uptake -> ~0.073 QALYs per person who completes screening.
  qaly_per_extra_completer = 0.073,
  # Programme cost GBP72m per 1m invitees at 59% uptake -> ~GBP121 per completer.
  cost_per_extra_completer = 121,
  threshold = 20000                  # GBP per QALY (paper); York calculator default is GBP15,000
)
STRATEGIES <- list(  # Asaria et al. (2016) Table 3
  universal = list(cost_per_letter = 3.50, uplift = 0.06, imd_reached = 1:5),
  targeted  = list(cost_per_letter = 7.00, uplift = 0.12, imd_reached = 1:2)   # paper: IMD4-5 = most deprived
)
# -----------------------------------------------------------------------------
inputs

### 2c. TODO: net health effect by IMD group

Both strategies are compared with **standard screening** (no reminder). For each IMD group:

- `extra_completers` = invitees × uplift, if the group receives the letter (else 0). *(pre-written)*
- **`qalys`** = extra_completers × `qaly_per_extra_completer` × need modifier × colonoscopy modifier, where each modifier is the group's value divided by the mean across the 5 groups. *(Higher bowel cancer burden, or better follow-up, means more health gained per extra person screened.)*
- **`total_cost`** (one number) = letter cost × invitees in reached groups + extra completers × `cost_per_extra_completer`.
- **`hoc`** (health opportunity cost) = the group's share of opportunity costs × `total_cost` / `threshold`.
- `net_qalys` = qalys − hoc. *(pre-written)*

Fill in the three lines marked `TODO`.

In [ ]:
net_health_by_imd <- function(strategy, inputs = get("inputs", envir = globalenv()), params = PARAMS, hoc_scenario = "flat") {
  s <- STRATEGIES[[strategy]]
  reached <- inputs$imd %in% s$imd_reached
  extra_completers <- inputs$invitees * s$uplift * reached
  need_mod <- inputs$need_share / mean(inputs$need_share)
  colo_mod <- inputs$colo_uptake / mean(inputs$colo_uptake)
  hs <- hoc_dist[hoc_dist$scenario == hoc_scenario, ]
  hoc_share <- hs$hoc_prop[order(hs$imd)]

  qalys <- NULL        # TODO
  total_cost <- NULL   # TODO
  hoc <- NULL          # TODO

  data.frame(imd = inputs$imd, extra_completers = extra_completers,
             qalys = qalys, hoc = hoc, net_qalys = qalys - hoc)
}

In [ ]:
tryCatch({
  r <- net_health_by_imd("targeted")
  stopifnot(abs(sum(r$qalys) - 3017.7) < 1, abs(sum(r$net_qalys) - 2587.3) < 1)
  cat("passed: targeted reminder, total net population QALYs =", round(sum(r$net_qalys), 1), "\n")
}, error = function(e) cat("not quite yet:", conditionMessage(e), "\n"))

### Solution: Section 2

In [ ]:
net_health_by_imd_solution <- function(strategy, inputs = get("inputs", envir = globalenv()), params = PARAMS, hoc_scenario = "flat") {
  s <- STRATEGIES[[strategy]]
  reached <- inputs$imd %in% s$imd_reached
  extra_completers <- inputs$invitees * s$uplift * reached
  need_mod <- inputs$need_share / mean(inputs$need_share)
  colo_mod <- inputs$colo_uptake / mean(inputs$colo_uptake)
  hs <- hoc_dist[hoc_dist$scenario == hoc_scenario, ]
  hoc_share <- hs$hoc_prop[order(hs$imd)]

  qalys <- extra_completers * params$qaly_per_extra_completer * need_mod * colo_mod
  total_cost <- sum(s$cost_per_letter * inputs$invitees * reached) +
                sum(extra_completers) * params$cost_per_extra_completer
  hoc <- hoc_share * total_cost / params$threshold

  data.frame(imd = inputs$imd, extra_completers = extra_completers,
             qalys = qalys, hoc = hoc, net_qalys = qalys - hoc)
}

results <- lapply(c(universal = "universal", targeted = "targeted"), net_health_by_imd_solution)
for (s in names(results)) {
  r <- results[[s]]
  cat(sprintf("%-10s gross QALYs %7.0f | opportunity cost %5.0f | NET %7.0f\n",
              s, sum(r$qalys), sum(r$hoc), sum(r$net_qalys)))
}

net_mat <- rbind(universal = results$universal$net_qalys, targeted = results$targeted$net_qalys)
barplot(net_mat, beside = TRUE, col = STRAT_COLOURS, names.arg = imd_labels,
        ylab = "Net population QALYs vs standard screening", main = "Who gains? Net health effect by deprivation group")
abline(h = 0)
legend("topright", fill = STRAT_COLOURS, legend = names(STRAT_COLOURS), bty = "n", y.intersp = 1.4)

> **Read this:** universal produces more health in total, but most of it goes to the less deprived groups (higher uptake, higher recorded bowel cancer burden). Targeted produces less in total, concentrated in IMD1–2, and the untargeted groups bear a small **net loss** because they still share the opportunity cost. That's the trade-off in one chart. Neither strategy dominates.

### 2d. First, the plain CEA answer (pre-built)

Before looking at *who* gains, here is what a standard cost-effectiveness analysis would conclude, using only the totals: the ICER (extra cost per extra QALY) and the net health benefit (QALYs gained minus the health lost elsewhere because of the spending).

In [ ]:
cea <- do.call(rbind, lapply(names(results), function(s) {
  r <- results[[s]]
  cost <- sum(r$hoc) * PARAMS$threshold            # total cost, recovered from the opportunity cost
  data.frame(strategy = s, extra_cost_GBP = round(cost), QALYs_gained = round(sum(r$qalys)),
             ICER_GBP_per_QALY = round(cost / sum(r$qalys)), net_health_benefit = round(sum(r$net_qalys)))
}))
print(cea, row.names = FALSE)

d_cost <- diff(rev(cea$extra_cost_GBP)); d_qaly <- diff(rev(cea$QALYs_gained))
cat(sprintf("\nUniversal vs targeted: +GBP%s for +%s QALYs -> ICER GBP%s per QALY\n",
            format(d_cost, big.mark = ","), format(d_qaly, big.mark = ","), format(round(d_cost / d_qaly), big.mark = ",")))

> **Plain CEA verdict:** both reminders are very good value (a few thousand pounds per QALY, far below the £20,000 threshold), and **universal** has the higher net health benefit. So standard CEA says *fund universal*. The rest of this notebook asks whether that answer changes once we care about **whose** health improves.

## Section 3: Fairness adjustment (~7 min), using the paper's data

The York data is split by deprivation only, so to practise **fairness adjustment** we switch to the paper's 20 subgroups (sex × area ethnic composition × IMD, Tables 2 and 5). This is the one section where you can check yourself against published numbers (Table 6).

The paper judges **sex**-related variation in health *fair* (not something policy should equalise) and **deprivation** and **ethnicity**-related variation *unfair*. Direct standardisation: regress QALE on all three (population-weighted), then predict everyone's QALE **as if they were male**. That removes the fair variation and keeps the unfair.

*(Paper coding here: IMD Q1 = least deprived.)*

In [ ]:
paper <- data.frame(
  sex = rep(c("male", "female"), each = 10),
  is_group = rep(rep(c("IS_Q1_4", "IS_Q5"), each = 5), 2),
  imd = rep(paste0("Q", 1:5), 4),
  population_pct = rep(c(6, 9, 10, 8, 6, 1, 2, 3, 2, 2), 2),
  qale_targeted = c(72.20, 70.52, 69.12, 66.63, 60.24, 72.21, 70.52, 69.13, 66.63, 60.24,   # Table 5
                    74.91, 73.16, 71.81, 69.24, 63.20, 74.92, 73.17, 71.82, 69.24, 63.20))
paper$male <- as.integer(paper$sex == "male")
paper$is_low <- as.integer(paper$is_group == "IS_Q1_4")   # reference = highest IS quintile
paper$imd <- factor(paper$imd)                            # reference level Q1 (least deprived)

# TODO: fit a population-weighted least-squares regression
#   formula: qale_targeted ~ male * imd + is_low
#   hint:    lm(formula, data = paper, weights = population_pct)
model <- NULL

In [ ]:
tryCatch({
  stopifnot(!is.null(model))
  b <- coef(model)
  cat(sprintf("passed: constant = %.3f (paper Table 6: 74.92), male = %.3f (paper: -2.708)\n", b["(Intercept)"], b["male"]))
  stopifnot(abs(b["(Intercept)"] - 74.92) < 0.05)
}, error = function(e) cat("not quite yet:", conditionMessage(e), "\n"))

### Solution: Section 3

In [ ]:
model_solution <- lm(qale_targeted ~ male * imd + is_low, data = paper, weights = population_pct)
print(round(coef(model_solution), 3))

as_if_male <- transform(paper, male = 1L)
paper$qale_fair_adjusted <- predict(model_solution, newdata = as_if_male)
paper[, c("sex", "is_group", "imd", "qale_targeted", "qale_fair_adjusted")]

> Compare with Table 6: constant 74.92, male −2.708, IMD Q5 −11.71. After adjustment every female row takes the male value for the same IMD and ethnicity group: the sex gap is gone, the deprivation gap remains. The adjustment encodes a **value judgement**, not a statistical fact. See Stretch B.

## Section 4: Dominance and inequality (~7 min)

Back to the York data. The post-intervention health of each IMD group is its baseline QALE plus its net population QALYs spread over its population:
`qale_post = qale + net_qalys / pop` (pre-built below).

The changes are tiny per person (a few millionths of a QALY averaged over 10.7 m people). That's normal: small programmes barely move lifetime health. So we look at **differences** from baseline, in population QALYs.

**Dominance.** One distribution *generalised-Lorenz dominates* another if, starting from the least healthy group, its cumulative total health is never lower. If the curves cross, dominance cannot rank the options and we need an inequality-aversion judgement (Section 5).

**TODO:** write `gini(h, p)`, the population-weighted Gini coefficient:

$$G = \frac{\sum_i \sum_j p_i\, p_j\, |h_i - h_j|}{2\,\bar h}, \qquad p_i = \text{population shares (sum to 1)},\; \bar h = \sum_i p_i h_i$$

*(Hint: `outer(h, h, "-")` gives every pairwise difference.)*

In [ ]:
pop  <- qale_imd$pop
base <- qale_imd$qale
post <- lapply(results, function(r) base + r$net_qalys / pop)

gini <- function(h, p) {
  # TODO: population-weighted Gini
  NULL
}

In [ ]:
tryCatch({
  g <- gini(base, pop)
  stopifnot(!is.null(g), abs(g - 0.033514) < 1e-5)
  cat(sprintf("passed: baseline Gini = %.5f\n", g))
}, error = function(e) cat("not quite yet:", conditionMessage(e), "\n"))

### Solution: Section 4

In [ ]:
gini_solution <- function(h, p) {
  w <- p / sum(p)
  sum(outer(w, w) * abs(outer(h, h, "-"))) / (2 * sum(w * h))
}

g0 <- gini_solution(base, pop)
cat(sprintf("Baseline Gini: %.6f\n", g0))
for (s in names(post)) cat(sprintf("%-10s change in Gini x 1,000,000: %+.2f   (total net QALYs %s)\n", s,
                                   (gini_solution(post[[s]], pop) - g0) * 1e6,
                                   format(round(sum(results[[s]]$net_qalys)), big.mark = ",")))

# Generalised Lorenz difference: cumulative net population QALYs, least -> most healthy group
ord <- order(base)                                    # IMD1 is least healthy
x <- c(0, cumsum(pop[ord]) / sum(pop))
plot(NA, xlim = c(0, 1), ylim = range(0, sapply(results, function(r) cumsum(r$net_qalys[ord]))),
     xlab = "Cumulative population share (least -> most healthy)", ylab = "Cumulative net population QALYs",
     main = "Generalised Lorenz (difference from standard screening)")
for (s in names(results)) lines(x, c(0, cumsum(results[[s]]$net_qalys[ord])), type = "o", pch = 19, col = STRAT_COLOURS[s], lwd = 2)
abline(h = 0)
legend("topleft", col = STRAT_COLOURS, lwd = 2, legend = names(STRAT_COLOURS), bty = "n")

> Targeted is **higher** at the bottom (more health for the least healthy 40%); universal is **higher** at the end (more total health). The curves **cross**, so neither dominates, just as in the paper. Note the Gini changes too: targeted *reduces* inequality, while universal slightly *increases* it, because its gains go mostly to the already-healthier groups.

## Section 5: Trading off total health against inequality (~9 min)

The **equally distributed equivalent (EDE)** is the level of health which, if everyone had it, society would value the same as the actual unequal distribution. The more inequality-averse we are, the further the EDE falls below the mean.

Atkinson (relative aversion $\varepsilon$): $\quad h_{ede} = \left[\sum_i p_i\, h_i^{\,1-\varepsilon}\right]^{1/(1-\varepsilon)}$ $\quad$ (geometric mean when $\varepsilon = 1$)

Kolm (absolute aversion $\alpha$): $\quad h_{ede} = -\frac{1}{\alpha}\,\ln\!\left(\sum_i p_i\, e^{-\alpha h_i}\right)$

Multiply an EDE *difference* by the population size to get **equity-weighted population QALYs**.

**TODO:** implement both.

In [ ]:
atkinson_ede <- function(h, p, epsilon) {
  p <- p / sum(p)
  # TODO (optional: handle epsilon == 1 with the geometric mean)
  NULL
}

kolm_ede <- function(h, p, alpha) {
  p <- p / sum(p)
  # TODO
  NULL
}

In [ ]:
tryCatch({
  a <- atkinson_ede(base, pop, 10.95); k <- kolm_ede(base, pop, 0.15)
  stopifnot(!is.null(a), abs(a - 68.386) < 0.01, !is.null(k), abs(k - 68.481) < 0.01)
  cat(sprintf("passed: baseline EDE %.3f (Atkinson, eps=10.95) and %.3f (Kolm, alpha=0.15) vs mean %.3f\n", a, k, mean_qale))
}, error = function(e) cat("not quite yet:", conditionMessage(e), "\n"))

### Solution: Section 5

In [ ]:
atkinson_ede_solution <- function(h, p, epsilon) {
  p <- p / sum(p)
  if (abs(epsilon - 1) < 1e-9) return(exp(sum(p * log(h))))
  sum(p * h^(1 - epsilon))^(1 / (1 - epsilon))
}

kolm_ede_solution <- function(h, p, alpha) {
  p <- p / sum(p)
  -log(sum(p * exp(-alpha * h))) / alpha
}

ede_gap <- function(ede, param, post_dist = post, p = pop) {
  # Universal minus targeted, in equity-weighted population QALYs
  (ede(post_dist$universal, p, param) - ede(post_dist$targeted, p, param)) * sum(p)
}

crossover <- function(ede, grid, ...) {
  vals <- sapply(grid, function(g) ede_gap(ede, g, ...))
  i <- which(diff(sign(vals)) != 0)
  if (length(i)) grid[i[1]] else NA
}

eps  <- seq(0, 20, length.out = 401)
alph <- seq(0.001, 0.3, length.out = 300)
atk <- sapply(eps,  function(e) ede_gap(atkinson_ede_solution, e))
klm <- sapply(alph, function(a) ede_gap(kolm_ede_solution, a))
eps_x <- crossover(atkinson_ede_solution, eps); alpha_x <- crossover(kolm_ede_solution, alph)

shade <- function(xs, ys) {
  polygon(c(xs, rev(xs)), c(pmax(ys, 0), rep(0, length(xs))), col = adjustcolor(STRAT_COLOURS["universal"], 0.2), border = NA)
  polygon(c(xs, rev(xs)), c(pmin(ys, 0), rep(0, length(xs))), col = adjustcolor(STRAT_COLOURS["targeted"], 0.2), border = NA)
}
options(repr.plot.width = 12, repr.plot.height = 4.5)
par(mfrow = c(1, 2))
plot(eps, atk, type = "l", lwd = 2, xlab = "Atkinson epsilon", ylab = "Universal - targeted (equity-weighted QALYs)",
     main = sprintf("Crossover at epsilon ~ %.2f", eps_x)); shade(eps, atk); abline(h = 0); abline(v = 10.95, col = "red", lty = 2)
legend("bottomleft", legend = c("England elicited epsilon = 10.95", "universal preferred", "targeted preferred"), y.intersp = 1.4,
       col = c("red", NA, NA), lty = c(2, NA, NA), fill = c(NA, adjustcolor(STRAT_COLOURS, 0.2)), border = NA, bty = "n", cex = 0.8)
plot(alph, klm, type = "l", lwd = 2, xlab = "Kolm alpha", ylab = "Universal - targeted (equity-weighted QALYs)",
     main = sprintf("Crossover at alpha ~ %.3f", alpha_x)); shade(alph, klm); abline(h = 0); abline(v = 0.15, col = "red", lty = 2)
par(mfrow = c(1, 1)); options(repr.plot.width = 8, repr.plot.height = 4.5)

cat(sprintf("At eps = 0      : universal ahead by %s QALYs (plain CEA)\n", format(round(ede_gap(atkinson_ede_solution, 0)), big.mark = ",")))
cat(sprintf("At eps = 10.95  : universal - targeted = %s\n", format(round(ede_gap(atkinson_ede_solution, 10.95)), big.mark = ",")))
cat(sprintf("At alpha = 0.15 : universal - targeted = %s\n", format(round(ede_gap(kolm_ede_solution, 0.15)), big.mark = ",")))

> **Headline:** plain CEA ($\varepsilon=0$) prefers **universal**. Once society's inequality aversion passes a moderate level, **targeted** is preferred, and at England's elicited aversion (ε≈10.95, α≈0.15; Robson et al. 2017) it wins clearly. The paper reached the same qualitative conclusion with different data (its crossover: ε≈8, α≈0.12).

### 5b. Sensitivity (pre-built): do the value judgements and data problems matter?

In [ ]:
scenarios <- list(
  list(label = "Base case (flat opportunity cost, GBP20k)", hoc = "flat", params = PARAMS, inp = inputs),
  list(label = "Moderate opportunity cost, GBP15k", hoc = "moderate", params = modifyList(PARAMS, list(threshold = 15000)), inp = inputs),
  list(label = "Steep opportunity cost, GBP15k", hoc = "steep", params = modifyList(PARAMS, list(threshold = 15000)), inp = inputs),
  list(label = "Equal need (ignore hospital-episode data)", hoc = "flat", params = PARAMS, inp = transform(inputs, need_share = 0.2))
)
sens <- do.call(rbind, lapply(scenarios, function(sc) {
  res <- lapply(c(universal = "universal", targeted = "targeted"), net_health_by_imd_solution,
                inputs = sc$inp, params = sc$params, hoc_scenario = sc$hoc)
  pst <- lapply(res, function(r) base + r$net_qalys / pop)
  gapE <- ede_gap(atkinson_ede_solution, 10.95, post_dist = pst)
  data.frame(scenario = sc$label, U_minus_T_eps0 = round(ede_gap(atkinson_ede_solution, 0, post_dist = pst)),
             crossover_eps = crossover(atkinson_ede_solution, eps, post_dist = pst),
             U_minus_T_eps10.95 = round(gapE), preferred_at_England_eps = ifelse(gapE < 0, "targeted", "universal"))
}))
sens

> In this example the ranking is **robust**: the opportunity-cost scenarios barely matter because the reminder programmes are cheap, and dropping the hospital-episode need data moves the crossover *earlier* (ε≈4), because the recorded bowel cancer burden is slightly higher in less deprived areas. With a costlier intervention the opportunity-cost assumption can flip the answer (paper Table 9). Try doubling `cost_per_extra_completer` in `PARAMS` and re-running Sections 2–5.

---

## Wrap-up: back to Jack and Jill and your poll answer

At the start of the session you answered a poll: *Jill can gain 1 healthy year; what is the smallest gain for Jack (most deprived fifth, QALE ≈ 63) at which you'd still choose Jack over Jill (least deprived fifth, QALE ≈ 75)?*

Your answer implies how much more a QALY for Jack is worth to you than one for Jill, and so an Atkinson inequality-aversion parameter: $\text{weight} = 12 / \text{months}$ and $\varepsilon = \ln(\text{weight}) / \ln(\text{QALE}_{Jill} / \text{QALE}_{Jack})$. This is a two-person approximation. Put your answer in the cell below and run it to see which letter *you* would fund.

In [ ]:
my_poll_answer_months <- 6       # <- your poll answer: 12, 6, 4, 3, 2 or 1 (months of healthy life for Jack)

jack <- qale_imd$qale[1]; jill <- qale_imd$qale[5]    # IMD1 (most deprived), IMD5 (least)
weight <- 12 / my_poll_answer_months
my_eps <- log(weight) / log(jill / jack)
my_gap <- ede_gap(atkinson_ede_solution, my_eps)

cat(sprintf("Your answer: Jack's healthy year is worth %.1fx Jill's  ->  epsilon = %.1f\n", weight, my_eps))
cat(sprintf("The crossover in this analysis is epsilon = %.2f; England's public: 10.95\n", eps_x))
cat(sprintf("At your epsilon, universal - targeted = %s equity-weighted QALYs  ->  you would fund the %s letter\n",
            format(round(my_gap), big.mark = ","), if (my_gap > 0) "UNIVERSAL" else "TARGETED"))

Be ready to answer in one sentence: **which letter did your poll answer pick, did that surprise you, and which assumption would you most want better data on?**

## Stretch A: Same question, different country and equity dimension (US income, Chetty et al. 2016)

DCEA can use any equity-relevant grouping. Here the baseline is **US life expectancy at age 40 by sex × household income percentile** (200 groups, 2001–2014, CC0 data).

⚠️ **Caveats:** this is life expectancy *at 40*, reported as expected **age at death** (e.g. 83), not quality-adjusted and not from birth, so levels are not comparable with England. We simply **borrow the English per-person effects** to show how a finer-grained, income-based baseline changes the picture. The US has no single health budget, so the opportunity-cost logic is illustrative only.

In [ ]:
us <- load_csv(CHETTY_URL, "chetty_table1.csv")
us_pop <- as.numeric(us$count); us_h <- as.numeric(us$le_agg)
cat(sprintf("%d groups | mean LE at 40: %.2f | Gini: %.5f | Atkinson EDE (eps=10.95): %.2f\n",
            nrow(us), weighted.mean(us_h, us_pop), gini_solution(us_h, us_pop), atkinson_ede_solution(us_h, us_pop, 10.95)))

# Map income quintiles to the English IMD groups (quintile 1 = poorest ~ IMD1) and apply
# each strategy's English per-capita net QALYs to everyone in that quintile.
us$quintile <- (us$pctile - 1) %/% 20 + 1
us_post <- lapply(results, function(r) us_h + (r$net_qalys / pop)[us$quintile])

vals <- sapply(eps, function(e) ede_gap(atkinson_ede_solution, e, post_dist = us_post, p = us_pop))
cx <- crossover(atkinson_ede_solution, eps, post_dist = us_post, p = us_pop)
plot(eps, vals, type = "l", lwd = 2, xlab = "Atkinson epsilon", ylab = "Universal - targeted (scaled: read the sign)",
     main = sprintf("US income baseline: crossover at epsilon ~ %.2f", cx))
abline(h = 0); abline(v = 10.95, col = "red", lty = 2)

**Discuss:** US income inequality in life expectancy at 40 is *smaller in relative terms* than England's deprivation gap in QALE at birth (compare the Gini values). What does that do to the crossover? (England: ε≈5.5; US: ε≈10.) And is it reasonable to apply an aversion parameter elicited from the English public to a US decision?

## Stretch B: Change the fairness judgement

In Section 3, treat **ethnicity** as the fair source of variation instead of sex: predict everyone's QALE at `is_low = 1`, keeping their actual sex and IMD. Which gaps remain? The paper (Table 10) shows that the preferred strategy can flip depending on which characteristics are judged unfair.

**Resources:** Asaria, Griffin & Cookson (2016) *Med Decis Making*; Cookson, Griffin, Norheim & Culyer (eds.) (2020) *Distributional Cost-Effectiveness Analysis: Quantifying Health Equity Impacts and Trade-Offs*, OUP, with spreadsheet exercises at york.ac.uk/che/equity/handbook; York Health Equity Impact Calculator: shiny.york.ac.uk/dceasimple; Chetty et al. (2016) *JAMA* 315:1750–66.